# platoGPT — a decoder-only transformer, from scratch

A character-level **decoder-only transformer** built from PyTorch primitives and trained, one character at a time, on the complete dialogues of **Plato** (Jowett's translation, ~1.2 MB). Following Andrej Karpathy's [*Let's build GPT: from scratch*](https://www.youtube.com/watch?v=kCc8FmEb1nY), one step past the [makemore](https://github.com/karpathy/makemore) series. The finished model has ~10.8M parameters.

## The data

A character-level model sees text as a stream of characters. No tokenizer, no words — just the raw symbols and the task of
predicting the next one.

In [1]:
with open('tiny_plato.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f'length of dataset in characters: {len(text):,}')
print(text[:350])

length of dataset in characters: 1,215,348
PERSONS OF THE DIALOGUE:

     Lysimachus, son of Aristides.
     Melesias, son of Thucydides.
     Their sons.
     Nicias, Laches, Socrates.


LYSIMACHUS: You have seen the exhibition of the man fighting in armour,
Nicias and Laches, but we did not tell you at the time the reason why my
friend Melesias and I asked you to go with us and see him. I


In [2]:
# every unique character the model will ever see
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(f'vocab size: {vocab_size}')


 !"'(),-.012345:;=?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]abcdefghijklmnopqrstuvwxyz—’“”
vocab size: 78


### Tokenizing

We map each character to an integer and back. This is the entire "tokenizer": a lookup table.

In [3]:
stoi = { c:i for i,c in enumerate(chars) }
itos = { i:c for i,c in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]          # string -> list of ints
decode = lambda l: ''.join([itos[i] for i in l]) # list of ints -> string

print(encode("Socrates"))
print(decode(encode("Socrates")))

[38, 62, 50, 65, 48, 67, 52, 66]
Socrates


In [4]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)

n = int(0.9 * len(data))   # 90% train, 10% validation
train_data = data[:n]
val_data = data[n:]
print(data.shape, data.dtype)

torch.Size([1215348]) torch.int64


### Batching

The model trains on chunks of `block_size` characters at a time. Within a single chunk of length $T$ there are actually
$T$ overlapping prediction problems — predict char 2 from char 1, char 3 from chars 1–2, and so on — which teaches the
transformer to make sense of contexts anywhere from 1 up to `block_size` characters long.

In [5]:
torch.manual_seed(1994)
block_size = 8   # (a small value here, just to illustrate; the real model uses 256)
batch_size = 4

def get_batch(split, block_size=block_size, batch_size=batch_size):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:', xb.shape)
print('targets:', yb.shape)
for t in range(block_size):
    print(f'context {xb[0,:t+1].tolist()}  ->  target {yb[0,t].item()}')

inputs: torch.Size([4, 8])
targets: torch.Size([4, 8])
context [0]  ->  target 0
context [0, 0]  ->  target 38
context [0, 0, 38]  ->  target 34
context [0, 0, 38, 34]  ->  target 22
context [0, 0, 38, 34, 22]  ->  target 37
context [0, 0, 38, 34, 22, 37]  ->  target 20
context [0, 0, 38, 34, 22, 37, 20]  ->  target 39
context [0, 0, 38, 34, 22, 37, 20, 39]  ->  target 24


## The mathematical trick at the heart of self-attention

A token should be informed by the tokens *before* it (never after — this is a causal, left-to-right model). The simplest
version of "look back" is to average every previous token's information. Doing that with a loop is clear but slow; the trick
is that the same averaging is a single **matrix multiply** by a lower-triangular matrix of weights.

In [6]:
# toy data: batch B, time T, channels C
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)

# version 1: average all previous tokens with an explicit loop
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xbow[b, t] = torch.mean(x[b, :t+1], 0)

# version 2: the same thing as one matrix multiply by a normalized lower-triangular matrix
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x

print('same result?', torch.allclose(xbow, xbow2))
print(wei)

same result? True
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])


Averaging is just one choice of weights. The key realization: those weights don't have to be uniform. If we start from
zeros, mask out the future with $-\infty$, and pass the rows through a **softmax**, we get *data-independent* weights that
still sum to 1 — and once the weights are allowed to *depend on the tokens themselves*, we have self-attention.

In [7]:
import torch.nn.functional as F

tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))  # a token can't attend to the future
wei = F.softmax(wei, dim=-1)                      # rows sum to 1
print(wei)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])


### A single head of self-attention

Each position's vector $x$ is sent through three learned linear maps to a **query** $q = W_q x$, a **key** $k = W_k x$, and a **value** $v = W_v x$. The score between positions $i$ and $j$ is the dot product $q_i \cdot k_j$; we scale it by $1/\sqrt{d_k}$ to hold its variance near 1 (so the softmax stays soft), mask out $j > i$ so no position reads from the future, softmax across $j$ into weights that sum to 1, and take the weighted sum of the values.

In [8]:
torch.manual_seed(1994)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

head_size = 16
key   = torch.nn.Linear(C, head_size, bias=False)
query = torch.nn.Linear(C, head_size, bias=False)
value = torch.nn.Linear(C, head_size, bias=False)

k = key(x)
q = query(x)
wei = q @ k.transpose(-2, -1) * head_size**-0.5   # (B,T,T) scaled affinities

tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
out = wei @ value(x)

print('output shape:', out.shape)
print('attention weights for one example (note: data-dependent now, and each row sums to 1):')
print(wei[0])

output shape: torch.Size([4, 8, 16])
attention weights for one example (note: data-dependent now, and each row sums to 1):
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6349, 0.3651, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2705, 0.1770, 0.5525, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2731, 0.1858, 0.2317, 0.3093, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2093, 0.1359, 0.3378, 0.1893, 0.1278, 0.0000, 0.0000, 0.0000],
        [0.1701, 0.1275, 0.1754, 0.1956, 0.1728, 0.1586, 0.0000, 0.0000],
        [0.1254, 0.1569, 0.0982, 0.0879, 0.1510, 0.1529, 0.2276, 0.0000],
        [0.0711, 0.0311, 0.2694, 0.1869, 0.0923, 0.0961, 0.1333, 0.1197]],
       grad_fn=<SelectBackward0>)


## Scaling up to the real model

Everything below is the model that was actually trained on Plato. The hyperparameters:

In [9]:
import torch.nn as nn

# --- hyperparameters (the trained checkpoint) ---
block_size = 256    # context length
n_embed    = 384    # embedding / residual-stream width
n_head     = 6      # attention heads per block
n_layer    = 6      # transformer blocks
dropout    = 0.2    # regularization during training

device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: mps


### The building blocks

A **Head** is the self-attention above, packaged as a module. **MultiHeadAttention** runs several heads in parallel over disjoint slices of the vector and concatenates their outputs. **FeedForward** is the per-position MLP — a linear layer that widens the vector to $4\times$ its length, a ReLU, and a linear layer back — the one place each position's own vector is nonlinearly reworked. A **Block** wires the two sublayers together with the pieces that make depth trainable: **residual connections** (`x = x + sublayer(x)`) and **layer normalization** applied *before* each sublayer (pre-norm).

In [10]:
class Head(nn.Module):
    """one head of self-attention"""
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        return wei @ self.value(x)

class MultiHeadAttention(nn.Module):
    """several heads of self-attention in parallel"""
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embed, n_embed)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    """a per-token MLP"""
    def __init__(self, n_embed):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """a transformer block: pre-norm self-attention, then a per-position MLP, each in a residual connection"""
    def __init__(self, n_embed, n_head):
        super().__init__()
        head_size = n_embed // n_head
        self.sa   = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embed)
        self.ln1  = nn.LayerNorm(n_embed)
        self.ln2  = nn.LayerNorm(n_embed)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

### The full model

Token embeddings say *what* each character is; position embeddings say *where* it sits. Their sum flows through the stack of blocks, a final **layer norm** on the residual stream (`ln_f`), and a linear head that scores every possible next character.

In [11]:
class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table    = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        self.blocks = nn.Sequential(*[Block(n_embed, n_head) for _ in range(n_layer)])
        self.ln_f   = nn.LayerNorm(n_embed)   # final layer norm on the residual stream
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        if targets is None:
            return logits, None
        B, T, C = logits.shape
        loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]              # never exceed the context window
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature      # focus on the last step
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [12]:
model = GPTLanguageModel().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'{n_params/1e6:.2f} M parameters')

10.80 M parameters


### Training

The model was trained with AdamW for 5,000 steps (`batch_size=64`, `block_size=256`, `lr=3e-4`), which takes it from a
random-guess loss of $\ln(78)\approx 4.36$ down to about **1.0** nat/character on held-out text — the point where the samples start to
look convincingly Platonic. Rather than retrain here (that runs on a GPU), we load the saved weights and pick up from there.

In [13]:
# training loop, for reference — set train=True with a GPU to reproduce
train = False
if train:
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    for step in range(5000):
        xb, yb = get_batch('train', block_size=block_size, batch_size=64)
        xb, yb = xb.to(device), yb.to(device)
        _, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
    torch.save(model.state_dict(), 'platoGPT.pt')
else:
    model.load_state_dict(torch.load('platoGPT.pt', map_location=device))
    print('loaded platoGPT.pt')

loaded platoGPT.pt


In [14]:
@torch.no_grad()
def estimate_loss(eval_iters=20):
    model.eval()
    out = {}
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split, block_size=block_size, batch_size=32)
            xb, yb = xb.to(device), yb.to(device)
            _, loss = model(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    return out

print(estimate_loss())

{'train': 0.8649037480354309, 'val': 0.977580726146698}


### Generating pseudo-Plato

Start from a single newline and let the model continue the text, sampling each next character with a little temperature and top-k to keep it coherent.

In [15]:
model.eval()
context = torch.zeros((1, 1), dtype=torch.long, device=device)
sample = decode(model.generate(context, max_new_tokens=500, temperature=0.85, top_k=60)[0].tolist())
print(sample)


mind.

PHAEDRUS: But if not, Gorgias, that is the evil (Buture) is not the
pain of falsehood, or which the sense of the law is more advantage; they
bring fulll them to be cleared without them; for both of the eyes, and
falsehood, as the line of the highest produced is called by any other
drops in the legislator of those who finds that the argument may be
created out the action of the temperate. Do you not assist again your
supposition, you will explain, Lysimachus, sense, despise to this to
be s
